# MMD Pretrain-to-Task Benchmark

This benchmark mirrors [`benchmark_wasserstein.ipynb`](../wasserstein/benchmark_wasserstein.ipynb) and [`apply_benchmark.ipynb`](../apply_benchmark.ipynb) but uses the [`MMDEmbedder`](../../../src/data_meta_map/mmd_embedder.py) to recommend a source dataset for transfer learning.

Pipeline:

1. Load the same image-classification datasets that were pre-trained against in [`logs/pretrain_to_task_logs/`](../../../logs/pretrain_to_task_logs/).
2. Compute the pairwise `MMD²` matrix between datasets, optionally using a frozen pretrained encoder so the comparison happens in semantic feature space.
3. For every target dataset, recommend the closest source as the pretrain.
4. Look up the matching downstream `test_acc` from the logs and tabulate it against the random / big-pretrain baselines.

In [ ]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

from data_meta_map import MMDEmbedder, datasets as dm_datasets
from data_meta_map.mmd.encoders import PretrainedEncoder, RawEncoder

torch.manual_seed(0)
np.random.seed(0)

DATA_ROOT = '../../data'
LOGS_PATH = Path(__file__).parent.parent.parent.parent / 'logs' / 'pretrain_to_task_logs' \
    if '__file__' in dir() else Path('../../../logs/pretrain_to_task_logs')
DATASET_NAMES = ['mnist', 'cifar10', 'cifar100', 'kmnist', 'letters']

## Helpers — reused from the Wasserstein benchmark

In [ ]:
def get_pretrained_results(dataset_names, path_to_logs=LOGS_PATH):
    pretrain2downstream_results = defaultdict(dict)
    for pretrain_name in dataset_names:
        for downstream_name in dataset_names:
            if downstream_name == 'imagenet':
                continue
            if downstream_name == pretrain_name:
                continue
            log_path = Path(path_to_logs) / f'{pretrain_name}-{downstream_name}.yaml'
            if not log_path.exists():
                continue
            with open(log_path, 'r') as f:
                test_acc = yaml.safe_load(f)['task']['test_acc']
                pretrain2downstream_results[pretrain_name][downstream_name] = test_acc
    return pretrain2downstream_results


def get_random_baseline(dataset_names, path_to_pretrained_logs=LOGS_PATH):
    pretrain2downstream_results = get_pretrained_results(dataset_names + ['imagenet'], path_to_pretrained_logs)
    random_performance = {}
    for name in dataset_names:
        choice = name
        while choice == name:
            choice = np.random.choice(dataset_names)
        random_performance[name] = {
            'accuracy': pretrain2downstream_results[choice].get(name, np.nan),
            'pretrain': choice,
        }
    return random_performance


def get_big_pretrain_baseline(dataset_names, path_to_pretrained_logs=LOGS_PATH):
    pretrain2downstream_results = get_pretrained_results(dataset_names + ['imagenet'], path_to_pretrained_logs)
    return {
        name: {
            'accuracy': pretrain2downstream_results['imagenet'].get(name, np.nan),
            'pretrain': 'imagenet',
        }
        for name in dataset_names
    }

## MMD-based source selection

We use a frozen pretrained ResNet-18 as the auxiliary feature extractor. This puts every dataset on the same semantic footing before computing MMD², exactly as the Wasserstein benchmark does with class statistics.

Switch `encoder=` to `RawEncoder()` to instead run MMD on raw flattened pixels (faster but a much weaker signal).

In [ ]:
def compute_mmd_distance_matrix(dataset_names, encoder, max_samples=500, kernel='rbf'):
    raw_datasets = [dm_datasets.__dict__[name](root=DATA_ROOT)[0] for name in dataset_names]
    embedder = MMDEmbedder(
        mode='distance',
        kernel=kernel,
        bandwidth='median',
        encoder=encoder,
        emb_dim=2,
        max_samples=max_samples,
        device='cpu',
        seed=0,
    )
    D = embedder.compute_pairwise_distances(raw_datasets).cpu().numpy()
    return D, dataset_names


def get_mmd_results(dataset_names, encoder, path_to_pretrained_logs=LOGS_PATH, **kwargs):
    D, names = compute_mmd_distance_matrix(dataset_names, encoder, **kwargs)
    pretrain2downstream_results = get_pretrained_results(names, path_to_pretrained_logs)

    method_performance = {}
    for i, target in enumerate(names):
        order = np.argsort(D[i])
        choice = None
        for idx in order:
            cand = names[idx]
            if cand != target and cand in pretrain2downstream_results and target in pretrain2downstream_results[cand]:
                choice = cand
                break
        method_performance[target] = {
            'accuracy': pretrain2downstream_results.get(choice, {}).get(target, np.nan),
            'pretrain': choice,
        }
    return method_performance, D, names

In [ ]:
pretrained_encoder = PretrainedEncoder(
    model_name='resnet18',
    pretrained=True,
    image_shape=(3, 224, 224),
    batch_size=32,
    device='cpu',
)
mmd_results, D, names = get_mmd_results(DATASET_NAMES, pretrained_encoder)
print('MMD² distance matrix:')
print(pd.DataFrame(D, index=names, columns=names).round(3))

In [ ]:
mmd_raw_results, _, _ = get_mmd_results(DATASET_NAMES, RawEncoder())
random_results = get_random_baseline(DATASET_NAMES)
big_results = get_big_pretrain_baseline(DATASET_NAMES)

## Results table

For each target dataset we report the chosen pretrain and the resulting downstream test accuracy. `MMD (ResNet-18)` and `MMD (raw)` are the two MMD variants, `Random` and `ImageNet` are baselines.

In [ ]:
def to_frame(results, label):
    rows = []
    for tgt, info in results.items():
        rows.append({
            'target': tgt,
            f'{label} pretrain': info['pretrain'],
            f'{label} acc': info['accuracy'],
        })
    return pd.DataFrame(rows).set_index('target')

summary = pd.concat([
    to_frame(mmd_results, 'MMD-ResNet'),
    to_frame(mmd_raw_results, 'MMD-raw'),
    to_frame(random_results, 'Random'),
    to_frame(big_results, 'ImageNet'),
], axis=1)
summary

In [ ]:
mean_acc = summary[[c for c in summary.columns if c.endswith(' acc')]].mean(axis=0)
print('Mean downstream accuracy across targets:')
print(mean_acc.round(4))

## Notes

* MMD is computed over `max_samples=500` items per dataset to keep the notebook CPU-friendly. Increasing this trades runtime for a tighter estimate.
* The `PretrainedEncoder` backend is the recommended setup for transfer-selection benchmarks: it puts every dataset in a common semantic space before MMD.
* The `RawEncoder` row is included as an ablation — raw-pixel MMD often produces near-degenerate distances on low-resolution datasets.
* For consistency with the other benchmarks in this directory the table reports test accuracy of an MLP head trained on top of the chosen ResNet-18 backbone, as logged by [`benchmarks/pretrain_benchmark/get_pretrained_to_task.py`](../get_pretrained_to_task.py).